# 21 — CES panel: the Part 8 switch model on pre-switch attitudes, and what recall does to switcher counts

Two checks raised by notebook 20 (ANES panel):

1. **Timing of the Part 8 ratios.** Notebook 11 estimates that cultural conservatism predicts leaving at ~1.8× economic views, immigration alone ~1.2×, using 2024 attitudes. The ANES panel showed coefficients inflate when attitudes are measured after the vote. Here the CES 2020–24 panel fits the same model to the **same people on the same items**, once with their 2020 answers and once with their 2024 answers, so only the timing differs.
2. **Recall contamination.** The CES 2024 cross-section defines switchers by recalled 2020 vote (`presvote20post`). The panel has both the 2020 post-election report (`CC20_410`) and the 2024 recall (`presvote20post_24`).

**Fixed before estimation.** Items identical in 2020 and 2024 (panel guide): economic — Medicare for all, drug-price negotiation, oppose ACA repeal (drug importation left out: not a left–right item); immigration — the four items of notebook 12; social — abortion as a choice (oppose), ban in all circumstances (support); policing — more police (support), fewer police (oppose). Cultural index = mean of the eight immigration, social and policing items, as in notebook 11. Model = notebook 11's `plus_controls` and `cultural_split` specifications (party ID, college, age, Hispanic, Black), weighted logit; party ID from 2020 in the primary comparison. Estimand: ratio of the cultural (and immigration) coefficient to the economic one, 2020 attitudes against 2024 attitudes; bootstrap 95% intervals, 1,000 resamples within outcome. The economic index here is health care only; the cross-section's eight economic items were not asked in 2020.

In [1]:
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf, warnings
from pathlib import Path
warnings.filterwarnings('ignore'); pd.set_option('display.width', 220)
RAW = Path('../data/raw'); TABLES = Path('../outputs/tables')
ITEMS = {  # name: (2020 var, 2024 var, code for the named direction)
    'm4a': ('CC20_327a', 'RC24_327a', 1), 'drug_neg': ('CC20_327b', 'RC24_327b', 1), 'keep_aca': ('CC20_327d', 'RC24_327c', 2),
    'no_legal_status': ('CC20_331a', 'CC24_331a', 2), 'border_patrol': ('CC20_331b', 'CC24_331b', 1), 'cut_legal_imm': ('CC20_331d', 'RC24_331c', 1), 'wall': ('CC20_331e', 'RC24_331d', 1),
    'no_abortion_choice': ('CC20_332a', 'RC24_332a', 2), 'ban_all_abortion': ('CC20_332f', 'RC24_332f', 1),
    'more_police': ('CC20_334c', 'RC24_334c', 1), 'no_fewer_police': ('CC20_334d', 'RC24_334d', 2)}
ECON = ['m4a', 'drug_neg', 'keep_aca']; IMM = ['no_legal_status', 'border_patrol', 'cut_legal_imm', 'wall']
SOC = ['no_abortion_choice', 'ban_all_abortion']; POL = ['more_police', 'no_fewer_police']
cols = [v for t in ITEMS.values() for v in t[:2]] + ['CC20_410', 'CC24_410', 'presvote20post_24', 'presvote20post_22', 'commonpostweight_24', 'commonweight_24',
        'pid7_20', 'pid7_24', 'educ_20', 'birthyr_20', 'race_20', 'hispanic_20']
w = pd.read_csv(RAW / 'ces_panel' / 'merged_recontact_2024_vv.csv', usecols=cols, low_memory=False)
w['wt'] = w.commonpostweight_24.fillna(w.commonweight_24)
for k, (a, b_, one) in ITEMS.items():
    for yy, v in (('20', a), ('24', b_)):
        x = pd.to_numeric(w[v], errors='coerce'); w[f'{k}_{yy}'] = (x == one).astype(float).where(x.isin([1, 2]))
for yy in ('20', '24'):
    for name, its in [('econ_left', ECON), ('immig_cons', IMM), ('social_cons', SOC), ('police_cons', POL), ('cult_cons', IMM + SOC + POL)]:
        w[f'{name}_{yy}'] = w[[f'{i}_{yy}' for i in its]].mean(axis=1)
    w[f'pid_{yy}'] = w[f'pid7_{yy}'].where(w[f'pid7_{yy}'] <= 7)
w['college'] = (w.educ_20 >= 5).astype(float); w['age'] = 2024 - w.birthyr_20
w['hisp'] = ((w.hispanic_20 == 1) | (w.race_20 == 3)).astype(float); w['black'] = ((w.race_20 == 2) & (w.hisp == 0)).astype(float)
def wm(x, ww):
    m = x.notna(); return np.average(x[m], weights=ww[m])
print(len(w), 'panelists')

6175 panelists


## 1. Recall of the 2020 vote

In [2]:
rc = pd.crosstab(w.CC20_410.fillna(-1).astype(int), w.presvote20post_24, margins=True)
print('2020 report (rows; -1 = no 2020 vote reported) vs 2024 recall (1 Biden 2 Trump 3-5 other 6 did not vote):'); print(rc)
true_sw = (w.CC20_410 == 1) & (w.CC24_410 == 2); rec_sw = (w.presvote20post_24 == 1) & (w.CC24_410 == 2)
out = {'true_switchers': int(true_sw.sum()), 'recall_switchers': int(rec_sw.sum()), 'in_both': int((true_sw & rec_sw).sum()),
       'recall_switchers_true_share': (true_sw & rec_sw).sum() / rec_sw.sum(), 'true_switchers_recalled_share': (true_sw & rec_sw).sum() / true_sw.sum(),
       'recall_switchers_reported_Trump_2020': int((rec_sw & (w.CC20_410 == 2)).sum())}
for lab, m in [('true', w.CC20_410 == 1), ('recall', w.presvote20post_24 == 1)]:
    s = w[m & w.CC24_410.isin([1, 2])]; out[f'switch_rate_{lab}'] = wm((s.CC24_410 == 2).astype(float), s.wt)
b = w[w.CC20_410 == 1]; out['biden_recall_agreement'] = (b.presvote20post_24 == 1).mean()
t = w[w.CC20_410 == 2]; out['trump_recall_biden'] = (t.presvote20post_24 == 1).mean()
recall = pd.Series(out); print(recall.round(4)); recall.to_csv(TABLES / 'ces_panel_recall_check.csv')
rc.to_csv(TABLES / 'ces_panel_recall_crosstab.csv')

2020 report (rows; -1 = no 2020 vote reported) vs 2024 recall (1 Biden 2 Trump 3-5 other 6 did not vote):
presvote20post_24     1     2   3   4   5    6   All
CC20_410                                            
-1                   51    28   1   0   0  457   537
1                  3005     7   0   0   1   51  3064
2                    19  2320   3   0   6   49  2397
4                     4     9  66  28  36    5   148
5                     0     4   0   0   1    9    14
6                     0     0   0   0   0    1     1
7                     2     9   1   1   0    1    14
All                3081  2377  71  29  44  573  6175
true_switchers                          39.0000
recall_switchers                        44.0000
in_both                                 34.0000
recall_switchers_true_share              0.7727
true_switchers_recalled_share            0.8718
recall_switchers_reported_Trump_2020     9.0000
switch_rate_true                         0.0406
switch_rate_recall          

In [3]:
# Do the misclassified change the switcher profile? (2024 attitudes, the cross-section's measurement)
mis = rec_sw & ~true_sw
rows = []
for c in ['econ_left_24', 'immig_cons_24', 'social_cons_24', 'police_cons_24', 'pid_24']:
    rows.append({'var': c, 'true_switchers': wm(w.loc[true_sw, c], w.loc[true_sw, 'wt']), 'recall_switchers': wm(w.loc[rec_sw, c], w.loc[rec_sw, 'wt']),
                 'misclassified': wm(w.loc[mis, c], w.loc[mis, 'wt']), 'loyal_Dem': wm(w.loc[(w.CC20_410 == 1) & (w.CC24_410 == 1), c], w.loc[(w.CC20_410 == 1) & (w.CC24_410 == 1), 'wt']),
                 'Trump_Trump': wm(w.loc[(w.CC20_410 == 2) & (w.CC24_410 == 2), c], w.loc[(w.CC20_410 == 2) & (w.CC24_410 == 2), 'wt'])})
prof = pd.DataFrame(rows).set_index('var'); print(prof.round(2)); print('misclassified N =', int(mis.sum()))
# reservoir share of switchers (immigration index >= 0.5 on 2024 answers) under each definition
for lab, m in [('true', true_sw), ('recall', rec_sw)]:
    print(f'{lab}: share of switchers immigration-conservative (2024 answers) = {wm((w.loc[m, "immig_cons_24"] >= 0.5).astype(float), w.loc[m, "wt"]):.2f}')
prof.to_csv(TABLES / 'ces_panel_recall_profile.csv')

                true_switchers  recall_switchers  misclassified  loyal_Dem  Trump_Trump
var                                                                                    
econ_left_24              0.61              0.51           0.47       0.92         0.37
immig_cons_24             0.74              0.75           0.80       0.34         0.85
social_cons_24            0.34              0.29           0.45       0.07         0.52
police_cons_24            0.58              0.69           0.99       0.48         0.90
pid_24                    3.53              3.51           5.27       1.79         5.86
misclassified N = 10
true: share of switchers immigration-conservative (2024 answers) = 0.88
recall: share of switchers immigration-conservative (2024 answers) = 0.88


## 2. The Part 8 switch model, same people and items, 2020 vs 2024 attitudes

In [4]:
bb = w[(w.CC20_410 == 1) & w.CC24_410.isin([1, 2])].copy(); bb['switch'] = (bb.CC24_410 == 2).astype(int)
def fit(d, yy, pid, split):
    X = [f'econ_left_{yy}'] + ([f'immig_cons_{yy}', f'social_cons_{yy}', f'police_cons_{yy}'] if split else [f'cult_cons_{yy}'])
    d = d.dropna(subset=X + [pid, 'age']).copy()
    for x in X: d[x[:-3] + '_z'] = (d[x] - d[x].mean()) / d[x].std()
    Z = [x[:-3] + '_z' for x in X]
    m = smf.glm('switch ~ ' + ' + '.join(Z) + f' + {pid} + college + age + hisp + black', data=d, family=sm.families.Binomial(),
                freq_weights=d.wt * len(d) / d.wt.sum()).fit(cov_type='HC1')
    return m, d, Z
SPECS = [('2020 attitudes, 2020 party ID', '20', 'pid_20'), ('2024 attitudes, 2020 party ID', '24', 'pid_20'), ('2024 attitudes, 2024 party ID', '24', 'pid_24')]
rng = np.random.default_rng(21); rows = []
for lab, yy, pid in SPECS:
    for split in (False, True):
        m, d, Z = fit(bb, yy, pid, split)
        bs = []
        for _ in range(1000):
            r = d.groupby('switch', group_keys=False).apply(lambda s: s.sample(len(s), replace=True, random_state=rng.integers(1e9)))
            try:
                mb, _, _ = fit(r, yy, pid, split); bs.append([mb.params[z] for z in Z] + [abs(mb.params[z] / mb.params[Z[0]]) for z in Z[1:]])
            except Exception: pass
        bs = np.array(bs)
        for j, z in enumerate(Z):
            rows.append({'spec': lab, 'model': 'split' if split else 'cultural', 'term': z, 'coef': m.params[z], 'se': m.bse[z],
                         'boot_lo': np.percentile(bs[:, j], 2.5), 'boot_hi': np.percentile(bs[:, j], 97.5),
                         'ratio_to_econ': abs(m.params[z] / m.params[Z[0]]) if j else np.nan,
                         'ratio_lo': np.percentile(bs[:, len(Z) + j - 1], 2.5) if j else np.nan, 'ratio_hi': np.percentile(bs[:, len(Z) + j - 1], 97.5) if j else np.nan,
                         'N': len(d), 'events': int(d.switch.sum())})
res = pd.DataFrame(rows); print(res.round(2).to_string(index=False)); res.to_csv(TABLES / 'ces_panel_switch_model_timing.csv', index=False)

                         spec    model          term  coef   se  boot_lo  boot_hi  ratio_to_econ  ratio_lo  ratio_hi    N  events
2020 attitudes, 2020 party ID cultural   econ_left_z -0.41 0.11    -0.84     0.10            NaN       NaN       NaN 1666      37
2020 attitudes, 2020 party ID cultural   cult_cons_z  1.32 0.15     0.82     2.02           3.24      1.42     48.34 1666      37
2020 attitudes, 2020 party ID    split   econ_left_z -0.42 0.12    -0.91     0.17            NaN       NaN       NaN 1666      37
2020 attitudes, 2020 party ID    split  immig_cons_z  0.76 0.13     0.27     1.54           1.83      0.45     27.22 1666      37
2020 attitudes, 2020 party ID    split social_cons_z  0.65 0.12     0.05     1.15           1.55      0.35     11.12 1666      37
2020 attitudes, 2020 party ID    split police_cons_z  0.38 0.15    -0.34     0.87           0.91      0.03     11.21 1666      37
2024 attitudes, 2020 party ID cultural   econ_left_z -0.72 0.12    -1.45     0.23         

## 3. Which index moved with the switch?

In [5]:
bb['grp'] = np.where(bb.switch == 1, 'switch', 'loyal')
rows = []
for k in ['econ_left', 'immig_cons', 'social_cons', 'police_cons', 'cult_cons']:
    r = {'index': k}
    for g in ['loyal', 'switch']:
        s = bb[bb.grp == g]; r[f'{g}_2020'] = wm(s[f'{k}_20'], s.wt); r[f'{g}_2024'] = wm(s[f'{k}_24'], s.wt)
    r['diff_in_change'] = (r['switch_2024'] - r['switch_2020']) - (r['loyal_2024'] - r['loyal_2020'])
    rows.append(r)
mv = pd.DataFrame(rows).set_index('index'); print(mv.round(3)); mv.to_csv(TABLES / 'ces_panel_index_change_by_path.csv')
sw = bb[bb.switch == 1]
for i in ECON: print(f'  switchers {i}: 2020 {wm(sw[i + "_20"], sw.wt):.2f} -> 2024 {wm(sw[i + "_24"], sw.wt):.2f}')

             loyal_2020  loyal_2024  switch_2020  switch_2024  diff_in_change
index                                                                        
econ_left         0.923       0.919        0.782        0.608          -0.170
immig_cons        0.168       0.344        0.500        0.741           0.064
social_cons       0.097       0.073        0.364        0.338          -0.001
police_cons       0.303       0.484        0.542        0.583          -0.140
cult_cons         0.184       0.311        0.477        0.601          -0.003
  switchers m4a: 2020 0.79 -> 2024 0.47
  switchers drug_neg: 2020 0.85 -> 2024 0.85
  switchers keep_aca: 2020 0.71 -> 2024 0.50


## Reading

See technical appendix I.10. In brief: measured before the switch the cultural-to-economic ratio is larger, not smaller, than measured after — economic views of the switchers moved right with the switch, which pulls the post-vote economic coefficient up. Immigration's ratio to economics is about the same either way. Recall contaminates the CES switcher group far less than the ANES one.